# Late-onset DCI: a walkthrough for clinicians

This notebook explains and runs the revised analysis of **factors associated with DCI after day 7** (revised analysis plan B, `revised_analysis_plan_B.md`).

**Why it was revised.** The editor and reviewers raised four problems with the first submission's logistic model:

| Concern | Fix in this analysis |
|---|---|
| Immortal-time bias: only patients who survive past day 7 *can* have late DCI | Day-7 landmark: follow only patients alive and DCI-free on day 7 |
| Overfitting (≈12 covariates, few events) | 8 prespecified covariates, 1 degree of freedom each |
| WFNS and Hunt–Hess collinear | WFNS only |
| Predictors changed between models (hypertension vs aspirin) | Stepwise adjustment and models with each exposure |
| No calibration | Bootstrap calibration, Brier score and C-index |

Each section: **what** we do, **why**, then the result. All output is aggregate; no patient-level data are shown.

## 1. Setup

The analysis code lives in `late_dci/`, split into layers: data access → cohort building → statistics → analyses. This notebook only calls the top layer (`analyses`) and the cohort builders.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')  # code/ on the path, as in the other notebooks
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ischemia_timing.late_dci import analyses
from ischemia_timing.late_dci.cohort import (CORE_COVARIATES, EXTENDED_COVARIATES, AnalysisSet, Event, FollowUpEnd,
                                             build_landmark_dataset, build_patients, build_piecewise_dataset, select)
from ischemia_timing.late_dci.data_sources import load_sources

pd.set_option('display.precision', 3)

# Readable labels for tables and plots
LABELS = {
    'age': 'Age (per year)', 'male': 'Male sex', 'hypertension': 'Hypertension', 'poor_wfns': 'WFNS 4-5',
    'fisher': 'Modified Fisher (per grade)', 'active_smoker': 'Active smoking', 'aspirin': 'Aspirin before bleed',
    'year': 'Calendar year (per year)', 'alcohol': 'Alcohol abuse', 'diabetes': 'Diabetes', 'statin': 'Statin',
    'oral_anticoagulation': 'Oral anticoagulation',
}

## 2. Data and cohort

Three sources are joined per patient:

- **DCI timings file**: verified DCI status and date/time of the first DCI image.
- **Registry**: baseline covariates and death.
- **Outcomes file**: used where the registry row is empty (mostly 2022–23): in-hospital death (discharge mRS 6) and missing ictus dates.
- **Ictus conflicts**: where the timings file and registry disagree (4 patients, typos such as 2003 for 2013), the registry date is used.

Time zero is the **ictus**. Example: DCI first imaged on day 9 at 12:00 → onset 9.5 days.

Two analysis sets:
- **Full cohort**: everyone with known DCI status, ictus, death status and discharge date. Used for cumulative incidence (no covariates needed).
- **Complete case**: additionally all 8 covariates known. Used for all regression models.

In [ ]:
patients = build_patients(load_sources())
full = select(patients, AnalysisSet.FULL_COHORT)
complete = select(patients, AnalysisSet.COMPLETE_CASE, CORE_COVARIATES)

full.flow.merge(complete.flow, on='step', how='right', suffixes=(' full cohort', ' complete case'))

## 3. The day-7 landmark: removing immortal-time bias

**The problem.** In the first submission, patients were classified as late DCI (onset > day 7) or not. A patient who died on day 4 was counted as "no late DCI", although they never had the chance to develop it. The comparison was therefore biased towards patients who survived.

**The fix.** A *landmark* analysis starts the clock on day 7 and includes only patients who are, on day 7:
- alive,
- DCI-free,
- still in hospital.

Everyone in the risk set has the same opportunity to develop late DCI. Follow-up ends at hospital discharge or day 21 (end of the usual surveillance window), whichever comes first.

```
day 0          day 7 (landmark)                  day 21
  |---------------|-----------------------------------|
  excluded:        risk set: followed until
  DCI, death,      - late DCI         (event)
  discharge        - death before DCI (competing event)
                   - discharge / day 21 (censored)
```

In [ ]:
landmark_full = build_landmark_dataset(full.patients)
landmark = build_landmark_dataset(complete.patients)
risk_set = landmark.dataset

landmark_full.flow.merge(landmark.flow, on='step', suffixes=(' full cohort', ' complete case'))

## 4. Cumulative incidence with a competing risk

**Why not Kaplan–Meier?** Kaplan–Meier treats death as censoring, i.e. as if dead patients could still develop DCI later. That overestimates the risk of DCI. Death *prevents* DCI; it is a **competing event**.

The **Aalen–Johansen** estimator gives the proportion of patients who actually had late DCI by each day, with death accounted for. The two curves (DCI, death) plus the event-free proportion always sum to 1.

In [ ]:
overall = analyses.cumulative_incidence(landmark_full.dataset)
by_wfns = analyses.cumulative_incidence(landmark_full.dataset, 'poor_wfns')

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].step(overall['time'] + 7, overall['cif_event'], where='post', label='Late DCI')
axes[0].step(overall['time'] + 7, overall['cif_competing'], where='post', label='Death before DCI')
axes[0].set_title(f'All patients at risk on day 7 (n = {len(landmark_full.dataset)})')

for group, curve in by_wfns.groupby('group'):
    label = 'WFNS 4-5' if group.endswith('=1') else 'WFNS 1-3'
    axes[1].step(curve['time'] + 7, curve['cif_event'], where='post', label=label)
axes[1].set_title('Late DCI by WFNS')

for ax in axes:
    ax.set_xlabel('Days after ictus'); ax.legend()
axes[0].set_ylabel('Cumulative incidence')
plt.tight_layout(); plt.show()

day21 = overall.iloc[-1]
print(f"By day 21: late DCI {day21['cif_event']:.1%}, death before DCI {day21['cif_competing']:.1%}")

## 5. Primary model: cause-specific Cox regression

**What it answers.** Among patients at risk on day 7, which baseline characteristics are associated with a higher *rate* of late DCI, while they are alive?

**Hazard ratio (HR).** HR 2 = at any given day, patients with the factor develop DCI at twice the rate of those without. HR < 1 = lower rate. For continuous variables the HR is per unit (e.g. per year of age).

**Cause-specific** means death simply ends follow-up. This is the standard choice for questions about *causes/associations*.

**Covariates** were fixed in advance (no stepwise selection), each using one degree of freedom, to keep ≈8 events per degree of freedom:

| Covariate | Coding |
|---|---|
| Age | per year |
| Sex | male vs female |
| Hypertension | yes / no |
| WFNS | 4–5 vs 1–3 (Hunt–Hess and GCS dropped: they measure the same thing) |
| Modified Fisher | per grade |
| Active smoking | yes / no |
| Aspirin before the bleed | yes / no |
| Calendar year | per year (proxy for changing monitoring and imaging) |

In [ ]:
primary = analyses.cause_specific_table(risk_set, CORE_COVARIATES, 'primary')
print(f"n = {primary['n'].iloc[0]}, late DCI events = {primary['events'].iloc[0]}")

def forest(table, title, hr='HR'):
    table = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(6, 0.45 * len(table) + 1))
    y = np.arange(len(table))
    ax.errorbar(table[hr], y, xerr=[table[hr] - table['lower'], table['upper'] - table[hr]], fmt='o', capsize=3)
    ax.axvline(1, color='grey', linestyle='--')
    ax.set_yticks(y); ax.set_yticklabels(table['covariate'].map(LABELS).fillna(table['covariate']))
    ax.set_xscale('log'); ax.set_xlabel(f'{hr} (95% CI, log scale)'); ax.set_title(title)
    plt.tight_layout(); plt.show()

forest(primary, 'Late DCI: cause-specific Cox, day-7 landmark')
primary[['covariate', 'HR', 'lower', 'upper', 'p']]

**How to read it.** A CI crossing 1 means the data are compatible with no association. Focus on the size and width of the interval, not on p < 0.05.

### Proportional hazards check

The Cox model assumes each HR is constant over follow-up. The Schoenfeld test checks this per covariate; a small p suggests the effect changes over time (see the day-5/day-10 landmarks in section 7).

In [ ]:
analyses.proportional_hazards_check(risk_set, CORE_COVARIATES)

## 6. Hypertension and aspirin

These two came out of the first submission (hypertension in the main model, aspirin in the Firth model). Carrying them over is a data-driven choice, so we treat them as **hypothesis-generating** and show how their estimates behave.

**Stepwise adjustment.** If an HR moves a lot when a covariate is added, that covariate was confounding it.

In [ ]:
analyses.sequential_adjustment(risk_set)

**Each exposure with and without the other.** Aspirin is often prescribed *because of* cardiovascular risk, including hypertension. If the two overlap, one can "steal" the other's effect depending on what is in the model.

In [ ]:
display(analyses.exposure_specific_models(risk_set)[['model', 'covariate', 'HR', 'lower', 'upper', 'p']])
analyses.hypertension_aspirin_association(risk_set)

## 7. Is the association specific to *late* DCI?

A factor associated with late DCI might just be associated with DCI in general. Two checks:

1. **Piecewise Cox model from ictus.** One model over the whole period, with effects allowed to differ before and after day 7. The *ratio* HR(after) / HR(before) ≈ 1 means the factor acts similarly early and late.
2. **Other landmarks (day 5 and day 10).** Day 7 is arbitrary; results should not hinge on it.

In [ ]:
analyses.piecewise_contrast(build_piecewise_dataset(complete.patients))

In [ ]:
tables = [analyses.cause_specific_table(build_landmark_dataset(complete.patients, landmark_day=day).dataset,
                                        CORE_COVARIATES, f'day {day:g}') for day in (5.0, 7.0, 10.0)]
landmarks = pd.concat(tables)
landmarks.pivot(index='covariate', columns='model', values='HR')[['day 5', 'day 7', 'day 10']]

## 8. Sensitivity analyses

Each one changes a single assumption. Results that hold across them are more trustworthy.

| Analysis | Question |
|---|---|
| Fine–Gray | Same factors when modelling the *proportion* who get DCI (death kept in the risk set)? |
| Ridge, full covariate set | Same picture with all original covariates, shrunk to avoid overfitting? |
| Censor at ICU discharge | Does reduced surveillance on the ward matter? |
| No day-21 cap | Do the very late DCIs change anything? |

**Fine–Gray vs cause-specific.** The cause-specific HR describes the *rate* of DCI in living patients. The Fine–Gray subdistribution HR (sHR) describes the *cumulative proportion* with DCI, which also reflects effects on death. Similar values mean death does not distort the picture.

In [ ]:
fine_gray = analyses.fine_gray_table(risk_set, CORE_COVARIATES)
forest(fine_gray, 'Late DCI: Fine-Gray', hr='sHR')

**Ridge regression** shrinks all coefficients towards zero. The amount of shrinkage is chosen by cross-validation. This allows including all original covariates without overfitting. CIs come from 200 bootstrap resamples. Clopidogrel is dropped (1 user in the risk set).

In [ ]:
complete_extended = select(patients, AnalysisSet.COMPLETE_CASE, EXTENDED_COVARIATES)
ridge = analyses.ridge_model(build_landmark_dataset(complete_extended.patients).dataset, EXTENDED_COVARIATES)
print('Dropped:', ridge.dropped_sparse)
forest(ridge.table, f'Late DCI: ridge Cox (penalizer {ridge.penalizer:g})')

In [ ]:
variants = {
    'hospital discharge, day 21 (primary)': build_landmark_dataset(complete.patients),
    'ICU discharge': build_landmark_dataset(complete.patients, follow_up_end=FollowUpEnd.ICU_DISCHARGE),
    'no day-21 cap': build_landmark_dataset(complete.patients, cap_day=None),
}
tables = pd.concat([analyses.cause_specific_table(v.dataset, CORE_COVARIATES, label) for label, v in variants.items()])
print(tables.groupby('model')[['n', 'events']].first())
tables.pivot(index='covariate', columns='model', values='HR')

## 9. Model check: calibration and discrimination

Reviewer 1 asked for calibration. The model's purpose is association, not individual prediction, so this is a check of model adequacy.

The model predicts each patient's risk of late DCI by day 21 (from two cause-specific models: DCI and death).

| Metric | Meaning | Ideal |
|---|---|---|
| Calibration slope | Are predicted risks too extreme (< 1) or too flat (> 1)? | 1 |
| Brier score | Mean squared error of predicted risk | 0 (lower = better) |
| C-index (Wolbers) | Probability that a patient with DCI had a higher predicted risk than one without | 0.5 = chance, 1 = perfect |

**Optimism correction.** A model always looks better on the data it was fitted on. We refit it on 200 bootstrap resamples, measure how much better each refit performs on its own sample than on the original data (the "optimism"), and subtract this. Takes ~30 s.

In [ ]:
check = analyses.model_check(risk_set, CORE_COVARIATES)
check.metrics

In [ ]:
calibration = check.calibration
limit = calibration[['predicted', 'observed']].max().max() * 1.1
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, limit], [0, limit], '--', color='grey', label='perfect')
ax.plot(calibration['predicted'], calibration['observed'], 'o-', label='quintiles of predicted risk')
ax.set_xlabel('Predicted risk of late DCI by day 21'); ax.set_ylabel('Observed (Aalen-Johansen)')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Summary (run of 2026-09-25)

- **Immortal time** is removed by the day-7 landmark: 313 complete-case patients at risk, 66 late DCI, 17 competing deaths.
- **Age**: older patients have a lower rate of late DCI (HR ≈ 0.97 per year), in every analysis.
- **Calendar year**: the strongest association (HR ≈ 1.19 per year), most plausibly reflecting better detection (perfusion imaging, neuromonitoring) rather than biology.
- **Hypertension**: no association (HR ≈ 0.89, CI 0.51–1.56). The first submission's finding does not hold once immortal time and overfitting are addressed.
- **Aspirin before the bleed**: HR ≈ 2, CI touching 1, stable across adjustment steps and sensitivity analyses. Based on 9 events in 24 users and the effect may vary over time → exploratory.
- **Early vs late**: no clear evidence that any factor acts differently before and after day 7; intervals are wide.
- **Model check**: acceptable calibration (corrected slope 0.84) and moderate discrimination (C-index 0.71).

Limitations: diagnostic pathway (clinical vs perfusion/neuromonitoring) not yet analysable; 2022–23 under-represented in complete-case models; changes in monitoring over 15 years only partly captured by calendar year.